# Лабораторная работа 6. Ансамбли (градиентный бустинг)

**Вариант 17** (нечётный). Набор данных Bankloan: прогноз кредитного дефолта.

Задачи:
1. Подготовка и предобработка данных
2. Обучение GradientBoostingClassifier с исходными параметрами
3. Оценка метрик и матрица ошибок
4. Подбор гиперпараметров через GridSearchCV
5. Исследование влияния learning_rate и числа деревьев


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 17
DATA_PATH = Path('Bankloan.csv')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')

## 1. Загрузка и предобработка

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Размер:', df.shape)
print(df.isna().sum())
print(df.head())

labeled = df.dropna().copy()
print('После удаления пропусков:', labeled.shape)
print(labeled['default'].value_counts(normalize=True))

X = labeled.drop(columns=['default'])
y = labeled['default'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
print(f'Train={len(X_train)}, Test={len(X_test)}')
print('Классы train:', dict(y_train.value_counts()))

# Масштабирование не обязательно для деревьев, но выполняется по заданию
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## 2. Базовая модель градиентного бустинга

In [ ]:
def metrics_dict(y_true, y_pred, y_score):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_score),
    }


base = GradientBoostingClassifier(random_state=RANDOM_STATE)
base.fit(X_train, y_train)
base_pred = base.predict(X_test)
base_score = base.predict_proba(X_test)[:, 1]
base_m = metrics_dict(y_test, base_pred, base_score)
print(pd.Series(base_m).round(4))
print(classification_report(y_test, base_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_test, base_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Нет дефолта', 'Дефолт'],
    yticklabels=['Нет дефолта', 'Дефолт'],
)
plt.xlabel('Прогноз')
plt.ylabel('Истина')
plt.title('Матрица ошибок: базовый GBM')
plt.tight_layout()
plt.savefig(FIG_DIR / 'cm_base.png', dpi=150)
plt.show()
print(cm)

## 3. Подбор гиперпараметров (GridSearchCV)

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.2],
    'min_samples_leaf': [1, 3],
}

grid = GridSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_train, y_train)
print('Лучшие параметры:', grid.best_params_)
print('Лучший F1 (CV):', round(grid.best_score_, 4))

best = grid.best_estimator_
best_pred = best.predict(X_test)
best_score = best.predict_proba(X_test)[:, 1]
best_m = metrics_dict(y_test, best_pred, best_score)
print(pd.Series(best_m).round(4))
print(classification_report(y_test, best_pred, digits=4))

In [ ]:
cm_best = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_best, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Нет дефолта', 'Дефолт'],
    yticklabels=['Нет дефолта', 'Дефолт'],
)
plt.xlabel('Прогноз')
plt.ylabel('Истина')
plt.title('Матрица ошибок: GBM после GridSearchCV')
plt.tight_layout()
plt.savefig(FIG_DIR / 'cm_best.png', dpi=150)
plt.show()
print(cm_best)

## 4. Влияние learning_rate и числа деревьев

In [ ]:
rates = [0.01, 0.05, 0.1, 0.2]
n_trees = [50, 100, 150, 200, 300]
rows = []
for lr in rates:
    for n in n_trees:
        model = GradientBoostingClassifier(
            n_estimators=n, learning_rate=lr, max_depth=3, random_state=RANDOM_STATE
        )
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rows.append({
            'learning_rate': lr,
            'n_estimators': n,
            'Accuracy': accuracy_score(y_test, pred),
            'F1': f1_score(y_test, pred),
        })

study = pd.DataFrame(rows)
display(study.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for lr in rates:
    part = study[study['learning_rate'] == lr]
    axes[0].plot(part['n_estimators'], part['Accuracy'], marker='o', label=f'lr={lr}')
    axes[1].plot(part['n_estimators'], part['F1'], marker='o', label=f'lr={lr}')
axes[0].set_xlabel('n_estimators')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs число деревьев')
axes[0].legend()
axes[1].set_xlabel('n_estimators')
axes[1].set_ylabel('F1')
axes[1].set_title('F1 vs число деревьев')
axes[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'lr_n_estimators.png', dpi=150)
plt.show()

## 5. Сводка

In [ ]:
summary = pd.DataFrame([
    {'Модель': 'Базовый GBM', **base_m},
    {'Модель': 'GBM + GridSearchCV', **best_m},
])
display(summary.round(4))
summary.to_csv('results_summary.csv', index=False)
study.to_csv('lr_study.csv', index=False)